<a href="https://colab.research.google.com/github/acapodanno/Large-Language-Model/blob/main/llm_esercitazione.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Esercitazione Sezione 4
Nel tutorial abbiamo visto come creare varie prompting strategy efficaci. In questa esercitazione dovrai pensare a un design della conversazione per un chatbot assistente per una SPA.

Considera questi dati:

Chatbot: L'esperienza nella SPA include il pacchetto "esperienza sensoriale". Non è previsto alcun costo aggiuntivo.
Chatbot: L'esperienza sensoriale in SPA è un viaggio di relax totale: ti immergi in un ambiente rilassante, avvolto da aromi di oli essenziali, mentre suoni naturali e luce soffusa creano un'atmosfera di pace. Include un massaggio con oli caldi e pietre levigate, che sciolgono le tensioni muscolari, accompagnato da un bagno caldo e avvolgente. Tutti i sensi vengono stimolati per favorire il massimo benessere.
User: A cosa servono gli oli essenziali?

Spiegazione esperienza:
I vari elementi della SPA sono studiati per offrirti un benessere completo, lavorando su corpo e mente.

Aromi: Gli oli essenziali, come lavanda o eucalipto, aiutano a rilassare la mente e migliorare l'umore. Inalando questi profumi, riduci lo stress e promuovi il rilassamento.

Massaggi: Servono a sciogliere le tensioni muscolari, migliorare la circolazione sanguigna e ridurre il dolore o la rigidità. Ti sentirai più leggero e disteso.

Calore (sauna, bagno turco, pietre calde): Il calore penetra in profondità nei muscoli, aumentando la circolazione e favorendo la disintossicazione del corpo. Aiuta anche a rilassare la mente.

Acqua (idromassaggio, vasche): Il movimento dell'acqua massaggia il corpo, riducendo lo stress fisico e migliorando il flusso linfatico, essenziale per la depurazione.

Suoni e luci soffuse: Creano un ambiente di calma, riducendo la stimolazione eccessiva della mente e permettendo una sensazione di pace e tranquillità.

Ogni elemento lavora insieme per farti sentire più rilassato, energizzato e rigenerato.

Utilizza questi dati per creare una prompting strategy che possa dare all'utente una risposta precisa alla sua domanda. Puoi usare gli LLM che preferisci.

In [26]:
!pip install langchain
!pip install langchain-openai

### Define Model

In [27]:
from langchain_openai import ChatOpenAI

OPENAI_API_KEY = ""

model = ChatOpenAI(openai_api_key=OPENAI_API_KEY,
                   model_name="gpt-3.5-turbo")

### Message History

In [28]:
chat_history = """
Chatbot: L'esperienza nella SPA include il pacchetto "esperienza sensoriale". Non è previsto alcun costo aggiuntivo.
Chatbot: L'esperienza sensoriale in SPA è un viaggio di relax totale: ti immergi in un ambiente rilassante, avvolto da aromi di oli essenziali, mentre suoni naturali e luce soffusa creano un'atmosfera di pace. Include un massaggio con oli caldi e pietre levigate, che sciolgono le tensioni muscolari, accompagnato da un bagno caldo e avvolgente. Tutti i sensi vengono stimolati per favorire il massimo benessere.
User: A cosa servono gli oli essenziali?
"""

In [29]:
experience ="""
 L'esperienza sensoriale in SPA è un viaggio di relax totale: ti immergi in un ambiente rilassante, avvolto da aromi di oli essenziali, mentre suoni naturali e luce soffusa creano un'atmosfera di pace. Include un massaggio con oli caldi e pietre levigate, che sciolgono le tensioni muscolari, accompagnato da un bagno caldo e avvolgente. Tutti i sensi vengono stimolati per favorire il massimo benessere.
"""

In [30]:
explain_experience = """
Spiegazione esperienza: I vari elementi della SPA sono studiati per offrirti un benessere completo, lavorando su corpo e mente.

Aromi: Gli oli essenziali, come lavanda o eucalipto, aiutano a rilassare la mente e migliorare l'umore. Inalando questi profumi, riduci lo stress e promuovi il rilassamento.

Massaggi: Servono a sciogliere le tensioni muscolari, migliorare la circolazione sanguigna e ridurre il dolore o la rigidità. Ti sentirai più leggero e disteso.

Calore (sauna, bagno turco, pietre calde): Il calore penetra in profondità nei muscoli, aumentando la circolazione e favorendo la disintossicazione del corpo. Aiuta anche a rilassare la mente.

Acqua (idromassaggio, vasche): Il movimento dell'acqua massaggia il corpo, riducendo lo stress fisico e migliorando il flusso linfatico, essenziale per la depurazione.

Suoni e luci soffuse: Creano un ambiente di calma, riducendo la stimolazione eccessiva della mente e permettendo una sensazione di pace e tranquillità.

"""

### Standard Response

In [31]:
user_response = """
Cosa sono gli oli essenziali??
"""




### PROMPT GOAL

In [32]:
PROMPT_GOAL = """
Sei un chat bot assistente per una SPA.
Stai avendo una conversazione con un cliente ```{chat_history}```
Cosa vuole sapee l'utente??```{user_response}```
"""

### PROMPT INTERACTION

In [33]:
PROMPT_INTERACTION = """
Sei un chat bot assistente per una SPA. Stai avendo una conversazione con un cliente.
Hai proposto un esperienza: ```{experience}```
{user_goal}
Genera uan risposta appropriata per continuare la conversazione nel modo piu semplice possibile.
La risposta alle domande sul esperienza sono queste :```{explain_experience}```
Messagi precedenti: ```{chat_history}```
"""

### PROMPT REFINE

In [34]:
PROMPT_REFINE = """
Sei un chat bot assistente per una SPA. Stai avendo una conversazione con un cliente.
La tua ultima rispsta é stata questa: ```{raw_response}```
Assicurati che la risposta sia professionale.
Elimina domande.
Menzioni gli oli essenziali.
"""

### Prompt Interaction

In [35]:
from langchain_core.prompts import PromptTemplate
prompt_start = PromptTemplate(
    input_variables=[
        "chat_history",
        "user_response"
    ],
    template=PROMPT_GOAL
)
promp_intro = PromptTemplate(
    input_variables=[
        "experience",
        "chat_history",
        "explain_experience"
    ],
    template=PROMPT_INTERACTION
)
prompt_refine = PromptTemplate(
    input_variables=["raw_response","explain_experience"],
    template=PROMPT_REFINE
)

explain = PromptTemplate.from_template(explain_experience)
chat_prompt = PromptTemplate.from_template(chat_history)
exp = PromptTemplate.from_template(experience)
user_resp = PromptTemplate.from_template(user_response)

### Chain

In [36]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
chain_one = prompt_start | model | StrOutputParser()
chain_two = promp_intro | model | StrOutputParser()
chain_three = prompt_refine | model | StrOutputParser()
final_chain =({
    "chat_history": chat_prompt,
    "user_response": user_resp,
    "experience": exp,
    "explain_experience": explain
}) | RunnablePassthrough.assign(
    user_goal = chain_one
)| RunnablePassthrough.assign(
    raw_response=chain_two
)| RunnablePassthrough.assign(
    finale_response=chain_three
)

result = final_chain.invoke({
    "chat_history": chat_history,
    "user_response": user_response,

})
print(result['finale_response'])

Gli oli essenziali sono uno degli elementi chiave della nostra esperienza benessere in SPA, grazie alle loro proprietà benefiche sulla salute e sul benessere. Possono essere utilizzati in vari modi, come massaggi sulla pelle, inalazione o diffusione nell'ambiente, per favorire la rilassazione, ridurre lo stress, migliorare l'umore e alleviare sintomi fisici. In questo modo, contribuiscono a creare un'atmosfera di totale relax durante il trattamento.
